# DKI: linear vs nonlinear fitness

Trains the **new** batched DKI model twice on the *same* data split and
seed, toggling only the fitness network:

* `nonlinear=False` (default) — fitness is `fcc2(fcc1(y))`, two stacked
  `Linear(N, N)` layers with no activation. Since `W₂(W₁y) = (W₂W₁)y`
  collapses to one linear map, the per-capita fitness is **linear**. This is
  the legacy `cNODE2` model.
* `nonlinear=True` — fitness is `fcc2(SiLU(fcc1(y)))` with hidden width
  `hidden_mult * N`. The SiLU makes the fitness itself **nonlinear**.

In both cases the dynamics are nonlinear via the replicator wrapper
`y * (out - <out, y>)`; the flag only changes the fitness function.

We compare train/val Bray-Curtis curves, best val BC, held-out test BC,
parameter count, and wall-clock per epoch.

## 1. Setup

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/metagenAu/DKI.git'
BRANCH   = 'claude/vigilant-cray-Cua6m'   # change to 'main' once merged
REPO_DIR = '/content/DKI'

# On Colab this clones the repo; locally, just run from the repo root.
if os.path.isdir('/content') and not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR])
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
sys.path.insert(0, os.getcwd())

import torch, numpy as np
from dki.device import auto_device
print('torch', torch.__version__, 'device', auto_device())

## 2. Data

Uses the bundled gLV synthetic data. Point `DATA_DIR` at any folder with a
`Ptrain.csv` (and optionally `Ptest.csv` / `Ztest.csv`) to use your own.

In [ ]:
from dki.data import load_dataset

DATA_DIR = os.path.join(os.getcwd(), 'data')
SEED = 0
VAL_FRACTION = 0.2

# Load once and reuse for both runs so the train/val split is identical.
data = load_dataset(DATA_DIR, val_fraction=VAL_FRACTION, seed=SEED)
print(f'n_species={data.n_species}  train={data.z_train.shape[0]}  '
      f'val={data.z_val.shape[0]}  '
      f"test={0 if data.z_test is None else data.z_test.shape[0]}")

## 3. Train both variants

Same data object, same seed, same optimizer settings (matching
`dki_colab.ipynb`: 1000 epochs, early-stop patience 200) — only
`nonlinear` differs. Lower `EPOCHS` if you just want a quick look.

In [ ]:
from dki.train import TrainConfig, train

EPOCHS = 1000          # both runs share this budget

def base_cfg(nonlinear):
    return TrainConfig(
        data_dir=DATA_DIR,
        out_dir=f"/tmp/dki_{'nonlinear' if nonlinear else 'linear'}",
        epochs=EPOCHS,
        batch_size=20,
        lr=1e-2,
        min_lr=1e-4,
        t_final=100.0,
        grad_clip=1.0,
        early_stop_patience=200,   # match the main notebook
        val_fraction=VAL_FRACTION,
        seed=SEED,
        nonlinear=nonlinear,
        hidden_mult=2,
        save_predictions=False,
    )

runs = {}
for label, nonlinear in [('linear', False), ('nonlinear', True)]:
    print(f'\n=== training {label} (nonlinear={nonlinear}) ===')
    model, result, run_data = train(base_cfg(nonlinear), data=data)
    n_params = sum(p.numel() for p in model.parameters())
    runs[label] = dict(model=model, result=result, data=run_data,
                       nonlinear=nonlinear, n_params=n_params)
    print(f'{label}: best val BC {result.best_val_loss:.4f} @ epoch '
          f'{result.best_epoch}  |  {n_params} params  |  '
          f'{np.mean(result.epoch_seconds):.3f}s/epoch')

## 4. Loss curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
colors = {'linear': '#1f77b4', 'nonlinear': '#d62728'}

for label, r in runs.items():
    res = r['result']
    axes[0].plot(res.train_loss, color=colors[label], label=label)
    axes[1].plot(res.val_loss, color=colors[label], label=label)
    axes[1].axhline(res.best_val_loss, color=colors[label], ls='--', alpha=0.5)

axes[0].set_title('Train Bray-Curtis'); axes[0].set_xlabel('epoch')
axes[0].set_ylabel('Bray-Curtis'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title('Val Bray-Curtis'); axes[1].set_xlabel('epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Head-to-head summary

Held-out **test** Bray-Curtis (lower is better), best val BC, parameter
count and per-epoch wall-clock.

In [ ]:
import pandas as pd
from dki.infer import predict
from dki.losses import bray_curtis

rows = []
for label, r in runs.items():
    res, d = r['result'], r['data']
    test_bc = np.nan
    if d.p_test is not None and d.z_test is not None:
        qtst = predict(r['model'], d.z_test, t_final=100.0)
        test_bc = bray_curtis(qtst.cpu(), d.p_test.cpu()).item()
    rows.append(dict(
        model=label,
        nonlinear=r['nonlinear'],
        best_val_bc=round(res.best_val_loss, 4),
        best_epoch=res.best_epoch,
        test_bc=round(test_bc, 4),
        n_params=r['n_params'],
        sec_per_epoch=round(float(np.mean(res.epoch_seconds)), 3),
    ))

summary = pd.DataFrame(rows).set_index('model')
summary

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
metrics = ['best_val_bc', 'test_bc']
x = np.arange(len(metrics)); w = 0.35
for i, label in enumerate(runs):
    vals = [summary.loc[label, m] for m in metrics]
    ax.bar(x + (i - 0.5) * w, vals, w, label=label, color=colors[label])
ax.set_xticks(x); ax.set_xticklabels(['best val BC', 'test BC'])
ax.set_ylabel('Bray-Curtis (lower is better)')
ax.set_title('Linear vs nonlinear fitness'); ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## Takeaways

* If the gLV data is well-described by linear per-capita fitness, the two
  curves track closely and the extra SiLU parameters buy little — the
  nonlinear model may even need more epochs to catch up.
* On data with genuine higher-order interactions, the nonlinear fitness
  should reach a lower val/test BC at the cost of more parameters and a
  slightly higher per-epoch time.
* Either way the per-epoch wall-clock stays small: the batched `dopri5`
  solve (one call over the whole minibatch, integrating only to the
  endpoints `[0, t_final]`) applies to both variants.